In [38]:
import pandas as pd
import geopandas as gpd

In [39]:
url = (
    "https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson?$limit=500"
)

df = gpd.read_file(url)

In [40]:
df.head()

,shape_area,ntaname,cdtaname,shape_leng,boroname,ntatype,nta2020,borocode,countyfips,ntaabbrev,cdta2020,geometry
0,35321808.4385,Greenpoint,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28919.5607283,Brooklyn,0,BK0101,3,047,Grnpt,BK01,"MULTIPOLYGON (((-73.93213 40.72816, -73.93238 ..."
1,28852852.8918,Williamsburg,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28134.0830277,Brooklyn,0,BK0102,3,047,Wllmsbrg,BK01,"MULTIPOLYGON (((-73.95814 40.7244, -73.95772 4..."
2,15208960.645,South Williamsburg,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),18250.2800908,Brooklyn,0,BK0103,3,047,SWllmsbrg,BK01,"MULTIPOLYGON (((-73.95024 40.70547, -73.94984 ..."
3,52267406.735,East Williamsburg,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),43184.8003755,Brooklyn,0,BK0104,3,047,EWllmsbrg,BK01,"MULTIPOLYGON (((-73.92406 40.71411, -73.92404 ..."
4,9982022.78755,Brooklyn Heights,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,14312.1922849,Brooklyn,0,BK0201,3,047,BkHts,BK02,"MULTIPOLYGON (((-73.99236 40.68969, -73.99436 ..."


In [41]:
df.isna().sum()

shape_area    0
ntaname       0
cdtaname      0
shape_leng    0
boroname      0
ntatype       0
nta2020       0
borocode      0
countyfips    0
ntaabbrev     0
cdta2020      0
geometry      0
dtype: int64

In [42]:
cols = df.columns.to_list()
cols.remove("geometry")
cols

['shape_area',
 'ntaname',
 'cdtaname',
 'shape_leng',
 'boroname',
 'ntatype',
 'nta2020',
 'borocode',
 'countyfips',
 'ntaabbrev',
 'cdta2020']

In [43]:
from pathlib import Path
import sys

In [44]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.database import get_engine
from sqlalchemy import text
import os
import getpass

os.environ["PGUSER"] = "sgurung"
os.environ["PGDATABASE"] = "citibike"
os.environ["PGPASSWORD"] = getpass.getpass("PostgreSQL password: ")

In [45]:
engine = get_engine()

with engine.connect() as conn:
    data = pd.read_sql(
        text(
            """
            Select *
            From station_neighborhoods
            """
    ),con=conn)

engine.dispose()

In [46]:
data.head()

,station_id,station_name,station_lat,station_lng,observed_as_start,observed_as_end,nta_code,nta_name,borough,nta_type,matched_neighborhood
0,1234.56,Morgan HCT Charging,40.708948,-73.932777,True,True,BK0104,East Williamsburg,Brooklyn,0,True
1,190 Morgan,190 Morgan,40.711072,-73.932096,True,True,BK0104,East Williamsburg,Brooklyn,0,True
2,1964.01,101 St & Fort Hamilton Pkwy,40.611240,-74.032820,True,True,BK1001,Bay Ridge,Brooklyn,0,True
3,2009.04,Shore Rd & 4 Ave,40.611500,-74.035130,True,True,BK1001,Bay Ridge,Brooklyn,0,True
4,2042.01,4 Ave & 99 St,40.613270,-74.033150,True,True,BK1001,Bay Ridge,Brooklyn,0,True


In [47]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2639 entries, 0 to 2638
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   station_id            2639 non-null   str    
 1   station_name          2639 non-null   str    
 2   station_lat           2639 non-null   float64
 3   station_lng           2639 non-null   float64
 4   observed_as_start     2639 non-null   bool   
 5   observed_as_end       2639 non-null   bool   
 6   nta_code              2541 non-null   str    
 7   nta_name              2541 non-null   str    
 8   borough               2541 non-null   str    
 9   nta_type              2541 non-null   str    
 10  matched_neighborhood  2639 non-null   bool   
dtypes: bool(3), float64(2), str(6)
memory usage: 330.1 KB


In [48]:
data.isna().sum()

station_id               0
station_name             0
station_lat              0
station_lng              0
observed_as_start        0
observed_as_end          0
nta_code                98
nta_name                98
borough                 98
nta_type                98
matched_neighborhood     0
dtype: int64

In [49]:
unmatched = data.loc[data["matched_neighborhood"] ==False]
unmatched.head()

,station_id,station_name,station_lat,station_lng,observed_as_start,observed_as_end,nta_code,nta_name,borough,nta_type,matched_neighborhood
99,JC065,Dey St,40.737711,-74.066921,False,True,NaN,NaN,NaN,NaN,False
786,JC140,Caven Point Recreation Fields,40.692220,-74.080250,False,True,NaN,NaN,NaN,NaN,False
1057,JC145,Mallory Ave & Roosevelt Ave,40.722030,-74.085960,False,True,NaN,NaN,NaN,NaN,False
1271,JC148,Bay St & Washington St,40.720216,-74.036560,False,True,NaN,NaN,NaN,NaN,False
1342,JC149,Washington St & Morgan St,40.719446,-74.036892,False,True,NaN,NaN,NaN,NaN,False


In [50]:
unmatched["observed_as_end"].value_counts()

observed_as_end
True     97
False     1
Name: count, dtype: int64

In [51]:
print(unmatched["station_id"].str[:2].value_counts())

station_id
JC    61
HB    33
LA     3
SY     1
Name: count, dtype: int64
